# [실습] ONNX/ONNX Runtime 적용 실습

## 📋 실습 개요

이 실습에서는 PyTorch/TensorFlow 모델을 ONNX 포맷으로 변환하고, ONNX Runtime을 사용하여 모델을 실행하고 최적화하는 방법을 학습합니다.

---

## 🎯 실습 목표

1. **PyTorch/TensorFlow 모델을 ONNX 포맷으로 변환하기**
2. **Google Colab에서 ONNX Runtime으로 모델 실행하기**
3. **최적화 프로바이더 적용 및 성능 향상 확인하기**

---

## 📝 실습 단계

### 1️⃣ Google Colab에서 필요 패키지 설치 (ONNX, ONNX Runtime)
### 2️⃣ PyTorch 모델을 ONNX 형식으로 변환
### 3️⃣ ONNX Runtime으로 모델 로드 및 최적화
### 4️⃣ GPU 최적화 프로바이더 적용 및 성능 측정

## ⚙️ 사전 준비: Google Colab 주의사항

```
⚠️ GPU 사용을 위해 런타임 유형을 'GPU'로 설정해야 합니다.

📌 설정 방법:
   런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU 선택
```

**주의할 점:**
- 무료 Colab GPU는 T4/K80으로 자원이 제한적이므로 큰 모델에는 OOM 오류가 발생할 수 있습니다
- 세션 간 GPU 메모리는 공유되므로 필요시 런타임 재시작이 필요할 수 있습니다

---

# 🔧 실습 1: 패키지 설치 및 환경 설정

## 필요한 패키지 설치

In [1]:
# ONNX 및 ONNX Runtime 설치
!pip install onnx onnxruntime-gpu

# PyTorch 설치 (이미 설치되어 있지 않은 경우)
!pip install torch torchvision

# 추가 유틸리티
!pip install numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 431.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 517.2 kB/s eta 0:00:00


## 패키지 임포트

In [2]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np
import time

print(f"PyTorch 버전: {torch.__version__}")
print(f"ONNX 버전: {onnx.__version__}")
print(f"ONNX Runtime 버전: {ort.__version__}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")

PyTorch 버전: 2.8.0+cu126
ONNX 버전: 1.19.1
ONNX Runtime 버전: 1.23.0
GPU 사용 가능: True


---

# 🧠 실습 2: PyTorch 모델을 ONNX 형식으로 변환

## 간단한 PyTorch 모델 정의

In [3]:
# 간단한 CNN 모델 예제
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * 56 * 56, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# 모델 인스턴스 생성
model = SimpleCNN()
model.eval()

print("PyTorch 모델 생성 완료!")
print(f"\n모델 구조:")
print(model)

PyTorch 모델 생성 완료!

모델 구조:
SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc): Linear(in_features=100352, out_features=10, bias=True)
)


## PyTorch → ONNX 변환

In [4]:
# 더미 입력 데이터 생성 (batch_size=1, channels=3, height=224, width=224)
dummy_input = torch.randn(1, 3, 224, 224)

# ONNX 파일 경로
onnx_path = "simple_cnn.onnx"

# 모델을 ONNX 형식으로 변환
torch.onnx.export(
    model,                          # 변환할 모델
    dummy_input,                    # 더미 입력
    onnx_path,                      # 저장할 파일 경로
    export_params=True,             # 모델 파라미터 저장
    opset_version=11,               # ONNX opset 버전
    do_constant_folding=True,       # 상수 폴딩 최적화
    input_names=['input'],          # 입력 이름
    output_names=['output'],        # 출력 이름
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"✅ ONNX 모델이 '{onnx_path}'에 저장되었습니다!")

/tmp/ipython-input-3924643545.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


✅ ONNX 모델이 'simple_cnn.onnx'에 저장되었습니다!


## ONNX 모델 검증

In [5]:
# ONNX 모델 로드 및 검증
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print("✅ ONNX 모델 검증 완료!")
print(f"\n📊 모델 정보:")
print(f"   - 입력: {onnx_model.graph.input[0].name}")
print(f"   - 출력: {onnx_model.graph.output[0].name}")
print(f"   - Opset 버전: {onnx_model.opset_import[0].version}")

✅ ONNX 모델 검증 완료!

📊 모델 정보:
   - 입력: input
   - 출력: output
   - Opset 버전: 11


---

# 🚀 실습 3: ONNX Runtime으로 모델 실행 및 최적화

## CPU에서 ONNX Runtime 실행

In [6]:
# CPU Execution Provider로 세션 생성
ort_session_cpu = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

# 입력 데이터 준비
input_data = np.random.randn(1, 3, 224, 224).astype(np.float32)

# 추론 실행
input_name = ort_session_cpu.get_inputs()[0].name
output_name = ort_session_cpu.get_outputs()[0].name

print(f"입력 이름: {input_name}")
print(f"출력 이름: {output_name}")
print(f"입력 형태: {ort_session_cpu.get_inputs()[0].shape}")
print(f"출력 형태: {ort_session_cpu.get_outputs()[0].shape}")

# CPU 성능 측정
print("\n⏱️ CPU 성능 측정 중...")
start_time = time.time()
for _ in range(100):
    ort_outputs_cpu = ort_session_cpu.run([output_name], {input_name: input_data})
cpu_time = (time.time() - start_time) / 100

print(f"✅ CPU 평균 추론 시간: {cpu_time*1000:.2f}ms")

입력 이름: input
출력 이름: output
입력 형태: ['batch_size', 3, 224, 224]
출력 형태: ['batch_size', 10]

⏱️ CPU 성능 측정 중...
✅ CPU 평균 추론 시간: 1.91ms


## GPU에서 ONNX Runtime 실행 (최적화)

In [7]:
# GPU Execution Provider로 세션 생성
try:
    ort_session_gpu = ort.InferenceSession(
        onnx_path,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    )

    # 현재 사용 중인 Execution Provider 확인
    print(f"사용 중인 Execution Provider: {ort_session_gpu.get_providers()}")

    # GPU 성능 측정
    print("\n⏱️ GPU 성능 측정 중...")
    start_time = time.time()
    for _ in range(100):
        ort_outputs_gpu = ort_session_gpu.run([output_name], {input_name: input_data})
    gpu_time = (time.time() - start_time) / 100

    print(f"✅ GPU 평균 추론 시간: {gpu_time*1000:.2f}ms")
    print(f"🚀 CPU 대비 속도 향상: {cpu_time/gpu_time:.2f}배")

except Exception as e:
    print(f"⚠️ GPU 실행 실패: {e}")
    print("   CPU ExecutionProvider로 대체됩니다.")
    gpu_time = None

사용 중인 Execution Provider: ['CUDAExecutionProvider', 'CPUExecutionProvider']

⏱️ GPU 성능 측정 중...
✅ GPU 평균 추론 시간: 7.46ms
🚀 CPU 대비 속도 향상: 0.26배


---

# 📊 실습 4: 성능 비교 및 분석

## PyTorch vs ONNX Runtime 성능 비교

In [8]:
# PyTorch 추론 성능 측정
model.eval()
input_tensor = torch.randn(1, 3, 224, 224)

print("⏱️ PyTorch CPU 성능 측정 중...")
with torch.no_grad():
    start_time = time.time()
    for _ in range(100):
        pytorch_output = model(input_tensor)
    pytorch_time = (time.time() - start_time) / 100

# 결과 비교
print("\n📈 성능 비교 결과:")
print(f"{'='*60}")
print(f"PyTorch (CPU):         {pytorch_time*1000:.2f}ms")
print(f"ONNX Runtime (CPU):    {cpu_time*1000:.2f}ms")
print(f"ONNX Runtime 성능 향상: {pytorch_time/cpu_time:.2f}배")

if gpu_time:
    print(f"ONNX Runtime (GPU):    {gpu_time*1000:.2f}ms")
    print(f"{'='*60}")
    print(f"CPU 대비 GPU 성능 향상:  {cpu_time/gpu_time:.2f}배")
    print(f"PyTorch 대비 GPU 성능 향상: {pytorch_time/gpu_time:.2f}배")

⏱️ PyTorch CPU 성능 측정 중...

📈 성능 비교 결과:
PyTorch (CPU):         11.99ms
ONNX Runtime (CPU):    1.91ms
ONNX Runtime 성능 향상: 6.29배
ONNX Runtime (GPU):    7.46ms
CPU 대비 GPU 성능 향상:  0.26배
PyTorch 대비 GPU 성능 향상: 1.61배


## 결과 정확도 검증

In [9]:
# PyTorch와 ONNX Runtime 출력 비교
input_np = input_tensor.numpy()
ort_output = ort_session_cpu.run([output_name], {input_name: input_np})[0]

# 차이 계산
difference = np.abs(pytorch_output.numpy() - ort_output)
max_diff = np.max(difference)
mean_diff = np.mean(difference)

print("🔍 정확도 검증:")
print(f"{'='*60}")
print(f"최대 차이: {max_diff:.6f}")
print(f"평균 차이: {mean_diff:.6f}")
print(f"{'='*60}")

if max_diff < 1e-5:
    print("✅ PyTorch와 ONNX Runtime 출력이 일치합니다!")
else:
    print("⚠️ 출력에 약간의 차이가 있습니다 (허용 범위 내)")

# 출력 샘플 비교
print(f"\nPyTorch 출력 (처음 5개): {pytorch_output.numpy()[0, :5]}")
print(f"ONNX RT 출력 (처음 5개): {ort_output[0, :5]}")

🔍 정확도 검증:
최대 차이: 0.000001
평균 차이: 0.000000
✅ PyTorch와 ONNX Runtime 출력이 일치합니다!

PyTorch 출력 (처음 5개): [-0.17560713  0.18331061 -0.16783233 -0.11945154 -0.11731306]
ONNX RT 출력 (처음 5개): [-0.17560789  0.18331082 -0.16783252 -0.11945086 -0.11731305]


---

# 🎓 추가 학습: 최적화 기법

## 그래프 최적화

In [10]:
# SessionOptions를 사용한 최적화
sess_options = ort.SessionOptions()

# 최적화 레벨 설정
# ORT_DISABLE_ALL: 최적화 없음
# ORT_ENABLE_BASIC: 기본 최적화
# ORT_ENABLE_EXTENDED: 확장 최적화
# ORT_ENABLE_ALL: 모든 최적화
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# 최적화된 세션 생성
ort_session_optimized = ort.InferenceSession(
    onnx_path,
    sess_options=sess_options,
    providers=['CPUExecutionProvider']
)

print("✅ 그래프 최적화가 적용된 세션 생성 완료!")

# 성능 측정
start_time = time.time()
for _ in range(100):
    ort_outputs_opt = ort_session_optimized.run([output_name], {input_name: input_data})
optimized_time = (time.time() - start_time) / 100

print(f"\n최적화 전 CPU 시간: {cpu_time*1000:.2f}ms")
print(f"최적화 후 CPU 시간: {optimized_time*1000:.2f}ms")
print(f"성능 향상: {cpu_time/optimized_time:.2f}배")

✅ 그래프 최적화가 적용된 세션 생성 완료!

최적화 전 CPU 시간: 1.91ms
최적화 후 CPU 시간: 1.98ms
성능 향상: 0.96배


## 동적 Quantization (양자화)

In [11]:
from onnxruntime.quantization import quantize_dynamic, QuantType

# 양자화된 모델 경로
quantized_model_path = "simple_cnn_quantized.onnx"

# 동적 양자화 적용
print("양자화 진행 중...")
quantize_dynamic(
    onnx_path,
    quantized_model_path,
    weight_type=QuantType.QUInt8
)

print(f"✅ 양자화된 모델이 '{quantized_model_path}'에 저장되었습니다!")

# 모델 크기 비교
import os
original_size = os.path.getsize(onnx_path) / (1024 * 1024)
quantized_size = os.path.getsize(quantized_model_path) / (1024 * 1024)

print(f"\n📦 모델 크기 비교:")
print(f"{'='*60}")
print(f"원본 모델:     {original_size:.2f} MB")
print(f"양자화 모델:   {quantized_size:.2f} MB")
print(f"압축률:        {(1 - quantized_size/original_size)*100:.1f}%")
print(f"{'='*60}")

양자화 진행 중...
✅ 양자화된 모델이 'simple_cnn_quantized.onnx'에 저장되었습니다!

📦 모델 크기 비교:
원본 모델:     3.85 MB
양자화 모델:   0.97 MB
압축률:        74.9%


## 양자화 모델 성능 측정

In [12]:
# 양자화된 모델 로드
ort_session_quantized = ort.InferenceSession(
    quantized_model_path,
    providers=['CPUExecutionProvider']
)

# 성능 측정
print("⏱️ 양자화 모델 성능 측정 중...")
start_time = time.time()
for _ in range(100):
    ort_outputs_quant = ort_session_quantized.run([output_name], {input_name: input_data})
quantized_time = (time.time() - start_time) / 100

print(f"\n📊 양자화 모델 성능 비교:")
print(f"{'='*60}")
print(f"원본 모델:     {cpu_time*1000:.2f}ms")
print(f"양자화 모델:   {quantized_time*1000:.2f}ms")
print(f"속도 향상:     {cpu_time/quantized_time:.2f}배")
print(f"{'='*60}")

# 정확도 비교
original_output = ort_outputs_cpu[0]
quantized_output = ort_outputs_quant[0]
accuracy_diff = np.abs(original_output - quantized_output)

print(f"\n🔍 양자화 정확도 분석:")
print(f"최대 차이: {np.max(accuracy_diff):.6f}")
print(f"평균 차이: {np.mean(accuracy_diff):.6f}")

⏱️ 양자화 모델 성능 측정 중...

📊 양자화 모델 성능 비교:
원본 모델:     1.91ms
양자화 모델:   4.73ms
속도 향상:     0.40배

🔍 양자화 정확도 분석:
최대 차이: 0.006969
평균 차이: 0.002749


---

# 💡 실습 정리

## 학습한 내용

1. ✅ PyTorch 모델을 ONNX 포맷으로 변환하는 방법
2. ✅ ONNX Runtime을 사용하여 모델을 실행하는 방법
3. ✅ CPU와 GPU Execution Provider를 활용한 성능 최적화
4. ✅ 그래프 최적화 및 양자화 기법 적용

## 주요 장점

- 🚀 **성능 향상**: ONNX Runtime은 다양한 하드웨어에서 최적화된 추론 제공
- 🔄 **프레임워크 독립성**: PyTorch, TensorFlow 등 다양한 프레임워크 지원
- ⚡ **배포 용이성**: 경량화된 런타임으로 프로덕션 환경에 적합
- 🎯 **다양한 최적화**: 그래프 최적화, 양자화 등 다양한 최적화 기법 지원

---

## 📚 참고 자료

- [ONNX 공식 문서](https://onnx.ai/)
- [ONNX Runtime 문서](https://onnxruntime.ai/)
- [PyTorch ONNX Export](https://pytorch.org/docs/stable/onnx.html)
- [ONNX Model Zoo](https://github.com/onnx/models)

---

## 🔍 추가 실습 과제

1. **다른 모델로 실습**: ResNet, MobileNet 등 사전 학습된 모델로 변환 실습
2. **배치 추론**: 다양한 배치 크기로 성능 측정
3. **모바일 배포**: ONNX 모델을 모바일 환경(ONNX Runtime Mobile)에서 실행
4. **TensorRT 연동**: NVIDIA TensorRT Execution Provider 활용
5. **실제 데이터 추론**: 이미지 분류, 객체 검출 등 실제 작업에 적용

---

**실습 완료! 수고하셨습니다! 🎉**